# 03 · Fine-tuning de LaMa

**Qué hace:** fine-tunea el modelo LaMa (big-lama, vía simple-lama-inpainting) sobre el dataset sintético de tríos, con pérdida L1 enmascarada; evalúa preentrenado vs fine-tuned en el split de test (PSNR completo y en zona enmascarada).

**Qué necesita:**
- Dataset sintético — Release `v1.0` (`lama_synthetic.zip`); o regenerarlo con el notebook 02
- Paquete `synthetic_degradation/` (viene en el repo)
- GPU (Google Colab)

**Qué deja escrito:** checkpoint en `ROOT/_out/checkpoints/lama_finetuned/lama_best.pth` (usa `montar_drive_opcional()` para persistir en Drive)

**Arranque:** primera celda de código.

## 0. Setup

In [ ]:
!pip -q install simple-lama-inpainting omegaconf hydra-core lpips scikit-image pandas
from colab_setup import aplicar_parches, preparar_repo, descargar_datos, montar_drive_opcional
aplicar_parches()
ROOT = preparar_repo()
DATA = descargar_datos("release:lama_synthetic", root=ROOT)

In [ ]:
# Imports y configuración global
%matplotlib inline

import os, sys, site, shutil, subprocess
from pathlib import Path
from typing import Optional, Sequence
import cv2
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import torch
from IPython.display import display

ImageFile.LOAD_TRUNCATED_IMAGES = True

WORK_DIR = ROOT / "_out"
WORK_DIR.mkdir(parents=True, exist_ok=True)

LAMA_REPO_DIR = Path('/content/lama')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'GPU disponible: {torch.cuda.is_available()}')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
LAMA_DATA      = DATA["release:lama_synthetic"]
LAMA_TRAIN_DIR = LAMA_DATA / "train"
LAMA_VAL_DIR   = LAMA_DATA / "val"
LAMA_TEST_DIR  = LAMA_DATA / "test"
assert LAMA_TRAIN_DIR.exists(), f"Falta el dataset en {LAMA_DATA}. Descárgalo del Release o regenéralo con el notebook 02."
print("Dataset LaMa:", {s: len(list((LAMA_DATA / s / "images").glob("*.png"))) for s in ("train","val","test")})

## 1. Dataset y visualización

In [ ]:
# Fase 2: las mascaras ya NO se calculan por diferencia. El dataset (degradada,
# gt, mascara) viene pre-generado de Fase 1. Sanity-check de correspondencia.
def sanity_check_split(split_dir: Path) -> int:
    imgs = sorted((split_dir / 'images').glob('*.png'))
    assert imgs, f"Sin imagenes en {split_dir/'images'}"
    for p in imgs:
        stem = p.stem
        assert (split_dir / 'gt' / f'{stem}.png').exists(),   f"Falta gt de {stem}"
        assert (split_dir / 'masks' / f'{stem}_mask.png').exists(), f"Falta mask de {stem}"
    return len(imgs)

for d in (LAMA_TRAIN_DIR, LAMA_VAL_DIR, LAMA_TEST_DIR):
    print(d.name, '->', sanity_check_split(d), 'trios OK')

In [ ]:
# Celda 10: Visualizar 3 máscaras generadas
sample_images = sorted((LAMA_TRAIN_DIR/'images').glob('*.png'))[:3]
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i, img_path in enumerate(sample_images):
    mask_path = LAMA_TRAIN_DIR / 'masks' / img_path.name.replace('.png', '_mask.png')
    img  = Image.open(img_path).convert('RGB')
    mask = Image.open(mask_path).convert('L')
    overlay = np.array(img.copy())
    overlay[np.array(mask) > 127] = [255, 0, 0]
    axes[i][0].imshow(img);                axes[i][0].set_title('Degradada');  axes[i][0].axis('off')
    axes[i][1].imshow(mask, cmap='gray');  axes[i][1].set_title('Máscara');    axes[i][1].axis('off')
    axes[i][2].imshow(overlay);            axes[i][2].set_title('Overlay');    axes[i][2].axis('off')
plt.tight_layout()
plt.show()
print('Verifica: la máscara (rojo) cubre las zonas dañadas visibles.')

## 2. Setup LaMa

In [ ]:
# Celda 11: Clonar LaMa oficial (advimman/lama)
import subprocess, sys
from pathlib import Path

LAMA_REPO_URL = 'https://github.com/advimman/lama.git'
LAMA_REPO_DIR = Path('/content/lama')

if not LAMA_REPO_DIR.exists():
    subprocess.run(['git', 'clone', LAMA_REPO_URL, str(LAMA_REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(LAMA_REPO_DIR), 'pull'], check=True)

# ── Instalación de dependencias LaMa con filtrado de versiones incompatibles ──
# requirements.txt pina versiones de 2020-2021 que no compilan en Python 3.12
# (scikit-image==0.17.2, scikit-learn==0.24.2, tensorflow, wldhx.yadisk-direct)
# Las sustituimos por versiones modernas compatibles o las omitimos si ya están instaladas.

# Paquetes a OMITIR (ya instalados en versión nueva, o irrelevantes para fine-tuning)
SKIP_PKG = {
    'scikit-image',    # ya instalado vía pip install anterior (scikit-image>=0.19)
    'scikit-learn',    # ya instalado en versión moderna
    'tensorflow',      # no necesario para fine-tuning con PyTorch
    'wldhx.yadisk-direct',  # servicio cloud ruso, no disponible
}

# Paquetes a REEMPLAZAR con versiones compatibles
OVERRIDES = {
    'albumentations':      'albumentations>=1.3.0',
    'kornia':              'kornia>=0.6.0',
    'pytorch-lightning':   'pytorch-lightning>=1.9.4,<2.0',
    'hydra-core':          'hydra-core>=1.1.0',
}

req_file = LAMA_REPO_DIR / 'requirements.txt'
to_install = []
for raw_line in req_file.read_text().splitlines():
    line = raw_line.strip()
    if not line or line.startswith('#'):
        continue
    # Nombre del paquete (antes de ==, >=, <=, !=, ~=)
    import re
    pkg_name = re.split(r'[=<>!~]', line)[0].strip().lower()
    if pkg_name in SKIP_PKG:
        print(f'  [skip]     {line}')
        continue
    elif pkg_name in OVERRIDES:
        print(f'  [override] {line}  →  {OVERRIDES[pkg_name]}')
        to_install.append(OVERRIDES[pkg_name])
    else:
        to_install.append(line)

# Deduplicar
to_install = list(dict.fromkeys(to_install))
print(f'\nInstalando {len(to_install)} paquetes compatibles de LaMa...')

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + to_install,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('[WARN] Algunos paquetes fallaron:')
    print(result.stderr[-2000:])
else:
    print('[OK] Dependencias LaMa instaladas correctamente')

print(f'\nLaMa repo: {LAMA_REPO_DIR}')
print('Contenido:', [p.name for p in list(LAMA_REPO_DIR.iterdir())[:8]])

In [ ]:
# Celda 12: Descargar pesos LaMa pre-entrenados (big-lama) — con validación de tamaño
import subprocess, sys, shutil, inspect
from pathlib import Path

LAMA_REPO_DIR    = Path('/content/lama')
LAMA_WEIGHTS_DIR = LAMA_REPO_DIR / 'big-lama'
LAMA_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
(LAMA_WEIGHTS_DIR / 'models').mkdir(parents=True, exist_ok=True)
LAMA_MODEL_PATH  = LAMA_WEIGHTS_DIR / 'models' / 'best.ckpt'

MIN_SIZE_MB = 50   # big-lama.pt válido pesa ~200 MB

def _needs_download():
    if not LAMA_MODEL_PATH.exists():
        return True
    size_mb = LAMA_MODEL_PATH.stat().st_size // (1024 * 1024)
    if size_mb < MIN_SIZE_MB:
        print(f'Archivo existente demasiado pequeño ({size_mb} MB) — re-descargando...')
        LAMA_MODEL_PATH.unlink()
        return True
    return False

if not _needs_download():
    print('Pesos big-lama ya disponibles:',
          LAMA_MODEL_PATH.stat().st_size // (1024*1024), 'MB')
else:
    # ── Estrategia 1: simple-lama-inpainting (descarga automática en caché) ──
    try:
        from simple_lama_inpainting import SimpleLama
        import torch
        print("Cargando big-lama via simple-lama-inpainting...")
        _lama = SimpleLama()

        # Buscar .pt en rutas de caché habituales
        _module_dir = Path(inspect.getfile(SimpleLama)).parent
        _candidates = [
            _module_dir / 'big-lama.pt',
            Path.home() / '.cache' / 'simple_lama_inpainting' / 'big-lama.pt',
            Path.home() / '.cache' / 'simple-lama-inpainting'  / 'big-lama.pt',
            Path('/root/.cache/simple_lama_inpainting/big-lama.pt'),
        ]
        _src = next((p for p in _candidates if p.exists()), None)

        if _src and _src.stat().st_size // (1024*1024) >= MIN_SIZE_MB:
            shutil.copy(_src, LAMA_MODEL_PATH)
            print(f'Copiado desde caché: {_src}')
        else:
            torch.save(_lama.model.state_dict(), LAMA_MODEL_PATH)
            print('Pesos guardados desde SimpleLama.model (state_dict)')

    except Exception as e1:
        print(f'simple-lama-inpainting falló ({e1})')
        # ── Estrategia 2: huggingface_hub ──
        try:
            from huggingface_hub import hf_hub_download
            _src = hf_hub_download(repo_id='smartywu/big-lama', filename='big-lama.pt')
            shutil.copy(_src, LAMA_MODEL_PATH)
            print(f'Descargado via HF Hub: {_src}')
        except Exception as e2:
            print(f'HF Hub falló ({e2})')
            # ── Estrategia 3: GitHub releases de simple-lama-inpainting ──
            _url = ('https://github.com/enesmsahin/simple-lama-inpainting'
                    '/releases/download/v0.1.0/big-lama.pt')
            print(f'Descargando desde: {_url}')
            subprocess.run(['wget', '-q', '--show-progress',
                            '-O', str(LAMA_MODEL_PATH), _url], check=True)

    final_mb = LAMA_MODEL_PATH.stat().st_size // (1024*1024)
    assert final_mb >= MIN_SIZE_MB, f'Descarga incompleta: {final_mb} MB'
    print(f'[OK] Pesos big-lama listos: {final_mb} MB')

In [ ]:
# Añadir LaMa al sys.path. Los parches de compatibilidad (basicsr/functional_tensor,
# PIL._typing._Ink, mode setter de Pillow) ya los aplicó aplicar_parches() en la primera celda.
if str(LAMA_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(LAMA_REPO_DIR))

print('Entorno LaMa listo')

## 3. Fine-tuning

In [ ]:
# Celda 14: Fine-tuning LaMa (bucle PyTorch sobre SimpleLama JIT)
# - Dataset: pares (imagen, máscara) en LAMA_TRAIN_DIR / LAMA_VAL_DIR
# - Modelo: SimpleLama.model (JITWrapper con forward(image, mask))
# - Loss: L1 aplicada solo en la zona enmascarada (pred*mask vs gt*mask)
# - Optimización: Adam lr=1e-4, 10 épocas, checkpoint al mejor val_l1

from simple_lama_inpainting import SimpleLama
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from synthetic_degradation import load_triple_arrays

# Checkpoint del fine-tuning en el entorno de trabajo (efímero en Colab).
# persistir en Drive: DRIVE = montar_drive_opcional(); LAMA_BEST_CKPT = DRIVE / "checkpoints" / "lama_finetuned" / "lama_best.pth"
LAMA_BEST_CKPT  = ROOT / "_out" / "checkpoints" / "lama_finetuned" / "lama_best.pth"
LAMA_BEST_CKPT.parent.mkdir(parents=True, exist_ok=True)

import os, sys
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

class LamaFineTuneDataset(Dataset):
    """Devuelve (masked, mask, gt) con gt = la vintage LIMPIA (de gt/)."""
    def __init__(self, images_dir, masks_dir, gt_dir, img_size: int = 256):
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.gt_dir     = Path(gt_dir)
        self.stems = [p.stem for p in sorted(self.images_dir.glob('*.png'))]
        self.img_size = img_size

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, i):
        deg, gt, mask = load_triple_arrays(
            self.images_dir, self.masks_dir, self.gt_dir, self.stems[i], self.img_size)
        deg_t  = torch.from_numpy(deg).permute(2, 0, 1).float() / 255.0   # [3,H,W]
        gt_t   = torch.from_numpy(gt).permute(2, 0, 1).float() / 255.0    # [3,H,W]
        mask_t = torch.from_numpy(mask).unsqueeze(0).float()              # [1,H,W] in {0,1}
        masked = deg_t * (1.0 - mask_t)                                   # hole to 0
        return masked, mask_t, gt_t

train_ds = LamaFineTuneDataset(LAMA_TRAIN_DIR/'images', LAMA_TRAIN_DIR/'masks', LAMA_TRAIN_DIR/'gt')
val_ds   = LamaFineTuneDataset(LAMA_VAL_DIR/'images',   LAMA_VAL_DIR/'masks',   LAMA_VAL_DIR/'gt')
print(f"train={len(train_ds)}  val={len(val_ds)}")
train_dl = DataLoader(train_ds, batch_size=4,  shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=2,  shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} imgs | Val: {len(val_ds)} imgs')
print(f'Steps/epoch: {len(train_dl)} | Device: {DEVICE}')

# Cargar modelo SimpleLama
simple_lama = SimpleLama()
model = simple_lama.model.to(DEVICE)

# SimpleLama es un JITWrapper: forward(image: Tensor, mask: Tensor) → Tensor
# Verificar firma antes de entrenar
import inspect
try:
    sig = str(model.forward)
except Exception:
    sig = 'JIT module'
print(f'Modelo forward signature: {sig[:120]}')

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.L1Loss()

FALLBACK_EPOCHS = 10
best_val_loss   = float('inf')

for epoch in range(FALLBACK_EPOCHS):
    # ── Entrenamiento ──
    model.train()
    train_loss = 0.0
    for masked_img, mask, gt in train_dl:
        masked_img = masked_img.to(DEVICE)   # [B, 3, H, W]
        mask       = mask.to(DEVICE)          # [B, 1, H, W]
        gt         = gt.to(DEVICE)            # [B, 3, H, W]

        # FIX: JITWrapper espera (image, mask) como argumentos separados
        pred = model(masked_img, mask)         # → [B, 3, H, W]
        loss = criterion(pred * mask, gt * mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # ── Validación ──
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for masked_img, mask, gt in val_dl:
            masked_img = masked_img.to(DEVICE)
            mask       = mask.to(DEVICE)
            gt         = gt.to(DEVICE)
            pred       = model(masked_img, mask)
            val_loss  += criterion(pred * mask, gt * mask).item()

    train_loss /= len(train_dl)
    val_loss   /= len(val_dl)
    print(f'Epoch {epoch+1:02d}/{FALLBACK_EPOCHS} | '
          f'train_l1={train_loss:.4f} | val_l1={val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), LAMA_BEST_CKPT)
        print(f'  → Checkpoint guardado (val_l1={val_loss:.4f})')

print(f'\nFine-tuning LaMa completado. Mejor val_l1={best_val_loss:.4f}')
print(f'Checkpoint: {LAMA_BEST_CKPT}')

## 4. Evaluación: pre-entrenado vs fine-tuned

In [ ]:
# Funcion de inferencia LaMa (pre-entrenado o fine-tuned)
from simple_lama_inpainting import SimpleLama

def run_lama_inference(
    image: Image.Image,
    mask: Image.Image,
    model_state_dict_path: Optional[Path] = None,
) -> Image.Image:
    """
    Ejecuta LaMa inpainting. Si model_state_dict_path es None, usa pesos
    pre-entrenados por defecto. Si se pasa una ruta .pth, carga ese state_dict.
    """
    lama = SimpleLama()
    if model_state_dict_path is not None:
        state = torch.load(model_state_dict_path, map_location=DEVICE)
        lama.model.load_state_dict(state, strict=False)
        lama.model.to(DEVICE).eval()
    return lama(image, mask).convert('RGB')


In [ ]:
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Checkpoint LaMa fine-tuneado (None -> solo se evalua el modelo pre-entrenado).
_lama_ft_ckpts = sorted((ROOT / "_out" / "checkpoints" / "lama_finetuned").glob("*.pth"))
LAMA_FINETUNE_CKPT = _lama_ft_ckpts[-1] if _lama_ft_ckpts else None
if LAMA_FINETUNE_CKPT is None:
    print('[WARN] No se encontro checkpoint LaMa fine-tuneado; solo se evalua pre-entrenado.')

def _load_png(p):
    return np.array(Image.open(p).convert('RGB'))

def _masked_psnr_ssim(pred, gt, mask_bool):
    full_psnr = psnr(gt, pred, data_range=255)
    full_ssim = ssim(gt, pred, channel_axis=2, data_range=255)
    if mask_bool.any():
        sel = mask_bool[..., None].repeat(3, axis=2)
        region_psnr = psnr(gt[sel], pred[sel], data_range=255)
    else:
        region_psnr = float('nan')
    return full_psnr, full_ssim, region_psnr

test_stems = [p.stem for p in sorted((LAMA_TEST_DIR/'images').glob('*.png'))]
rows = []
for stem in test_stems:
    deg  = Image.open(LAMA_TEST_DIR/'images'/f'{stem}.png').convert('RGB')
    gt   = _load_png(LAMA_TEST_DIR/'gt'/f'{stem}.png')
    mask_img = Image.open(LAMA_TEST_DIR/'masks'/f'{stem}_mask.png').convert('L')
    mask_bool = np.array(mask_img.resize(gt.shape[1::-1])) > 127

    pred_pre = np.array(run_lama_inference(deg, mask_img).resize(gt.shape[1::-1]))
    p_full, s_full, p_reg = _masked_psnr_ssim(pred_pre, gt, mask_bool)
    row = {'stem': stem, 'pre_psnr': p_full, 'pre_ssim': s_full, 'pre_psnr_mask': p_reg}
    if LAMA_FINETUNE_CKPT is not None:
        pred_ft = np.array(run_lama_inference(deg, mask_img, LAMA_FINETUNE_CKPT).resize(gt.shape[1::-1]))
        f_full, fs_full, f_reg = _masked_psnr_ssim(pred_ft, gt, mask_bool)
        row.update({'ft_psnr': f_full, 'ft_ssim': fs_full, 'ft_psnr_mask': f_reg})
    rows.append(row)

import pandas as pd
df = pd.DataFrame(rows)
print(df.describe())
df.head()

In [ ]:
import matplotlib.pyplot as plt
show = test_stems[:3]
ncol = 4 if LAMA_FINETUNE_CKPT is not None else 3
fig, axes = plt.subplots(len(show), ncol, figsize=(4*ncol, 4*len(show)))
if len(show) == 1:
    axes = axes[None, :]
for r, stem in enumerate(show):
    deg = Image.open(LAMA_TEST_DIR/'images'/f'{stem}.png').convert('RGB')
    mask_img = Image.open(LAMA_TEST_DIR/'masks'/f'{stem}_mask.png').convert('L')
    gt  = Image.open(LAMA_TEST_DIR/'gt'/f'{stem}.png').convert('RGB')
    cols = [(deg, 'Degradada'), (run_lama_inference(deg, mask_img), 'LaMa pre')]
    if LAMA_FINETUNE_CKPT is not None:
        cols.append((run_lama_inference(deg, mask_img, LAMA_FINETUNE_CKPT), 'LaMa fine-tuned'))
    cols.append((gt, 'GT (limpia)'))
    for c, (im, title) in enumerate(cols):
        axes[r][c].imshow(im); axes[r][c].set_title(title); axes[r][c].axis('off')
plt.tight_layout(); plt.show()

## 5. Conclusiones

El checkpoint fine-tuneado queda en `ROOT/_out/checkpoints/lama_finetuned/lama_best.pth`
(usa `montar_drive_opcional()` para persistirlo en Drive), listo para la evaluación
end-to-end del pipeline (notebook 06).